In [1]:
API_KEY = ""

## 3.1 Anthropic messages api 호출 예시

In [ ]:
import anthropic

client = anthropic.Anthropic(
    api_key=API_KEY,
)

message = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=100,
    temperature=1.00,
    system="주어진 질문에 대해 간단하고 명확한 답변을 제공해주세요.",
    messages=[
        {
            "role": "user", 
            "content": "안녕하세요. 오늘 날씨가 어때요?"
        }
    ]
)

## 3.2 Anthropic messages api 호출 결과 예시

```json
{
    "id": "msg_01RMJEasZ2HkmUekTeZwr4dG",
    "content": [
        {
            "text": "죄송하지만 저는 실시간 날씨 정보를 가지고 있지 않습니다. 정확한 날씨는 기상청이나 날씨 앱을 통해 확인하시는 것이 좋겠습니다.",
            "type": "text"
        }
    ],
    "model": "claude-3-5-sonnet-20241022",
    "role": "assistant",
    "stop_reason": "end_turn",
    "stop_sequence": null,
    "type": "message",
    "usage": {
        "input_tokens": 60,
        "output_tokens": 82
    }
}
```

## 3.3 Ollama 설치 확인

In [ ]:
!ollama --version

## 3.4 Ollama 를 통한 Mistral 다운로드

In [ ]:
!ollama run mistral

In [ ]:
!ollama ps

## 3.5 Ollama chat api 호출 예시

In [ ]:
import requests

response = requests.post(
    "http://localhost:11434/api/chat",
    headers={"Content-Type": "application/json"},
    json={
        "model": "mistral",
        "messages": [
            {
                "role": "system",
                "content": "주어진 질문에 대해 간단하고 명확한 답변을 제공해주세요."
            },
            {
                "role": "user",
                "content": "안녕하세요. 오늘 날씨가 어때요?"
            }
        ],
        "options": {
            "temperature": 1.00,
            "num_predict": 100
        },
        "stream": False
    }
)


if response.status_code == 200:
    print(response.json())


## 3.6 Ollama chat api 호출 결과 예시

```json
{
    "model": "mistral",
    "created_at": "2024-12-01T04:58:28.684038Z",
    "message": {
        "role": "assistant",
        "content": "안녕하세요! 오늘의 날씨는 현재 알 수 없습니다. 날씨 정보를 확인하려면 지역을 명시하여 확인해 주시길 바랍니다."
    },
    "done_reason": "stop",
    "done": true,
    "total_duration": 2584408084,
    "load_duration": 12654292,
    "prompt_eval_count": 62,
    "prompt_eval_duration": 453000000,
    "eval_count": 74,
    "eval_duration": 2085000000
}
```

## 3.7 langchain 을 이용한 간단한 애플리케이션

In [ ]:
!pip install langchain -qq

In [ ]:
!pip install langchain-ollama -qq

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="mistral", temperature=0.01, max_tokens=100)


prompt_template = ChatPromptTemplate.from_messages([
    ("system", "주어진 질문에 대해 간단하고 명확한 답변을 제공해주세요."), 
    ("user", "{text}")
])

chain = prompt_template | llm | StrOutputParser()

chain.invoke({"text": "안녕하세요. 오늘 날씨가 어때요?"})

## 3.8 기존 코드에서 모델 변경하는 예시

In [ ]:
!pip install langchain-anthropic -qq

In [ ]:
import os
from langchain_anthropic import ChatAnthropic

os.environ["ANTHROPIC_API_KEY"] = API_KEY

anthropic_llm = ChatAnthropic(model='claude-3-5-sonnet-20241022')
chain = prompt_template | anthropic_llm | StrOutputParser()
chain.invoke({"text": "안녕하세요. 오늘 날씨가 어때요?"})

## 3.9 Langchain 파이썬 SDK 설치

In [ ]:
!pip install langchain langchain-ollama langchain-anthropic -qq

## 3.10 Langchain 기본 컴포넌트 임포트

In [4]:
from langchain_core.prompts import ChatPromptTemplate # 프롬프트 템플릿
from langchain_ollama import ChatOllama # Ollama chat API 사용
from langchain_core.output_parsers import StrOutputParser # 응답 텍스트 파싱

## 3.11 문장 감정 분류 샘플 애플리케이션

In [5]:
# 문장 감정 분류 프롬프트 템플릿화
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "주어진 문장의 감정을 긍정, 부정, 중립으로만 분류합니다. 분류에 대한 추가 설명을 하지 않습니다."), 
    ("user", "문장: {sentence}\n분류:")
])

# 모델 선언
llm = ChatOllama(model="mistral", temperature=0.1, max_tokens=100)

# 모델의 text 응답만 파싱하여 문자열로 변환
parser = StrOutputParser()

# 파이프 연산자를 통해 Chain 선언
chain = prompt_template | llm | parser

## 3.12 문장 감정 분류 애플리케이션 실행

In [6]:
sentences = [
    "음식은 괜찮았던 것 같아요.",
    "서비스가 별로였어요.",
    "매장이 깔끔하고 좋았어요."
]

for sentence in sentences:
    print(chain.invoke({"sentence": sentence}))

긍정
부정
긍정


## 3.13 String PromptTemplates 예시

In [7]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template("{topic}에 대한 농담을 해줘")
prompt_template.invoke({"topic": "고양이"})

StringPromptValue(text='고양이에 대한 농담을 해줘')

## 3.14 Chat PromptTemplates 예시

In [8]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate([
    ("system", "당신은 주어진 요청에 도움을 주는 조수입니다."),
    ("user", "{topic}에 대한 농담을 해줘")
])

prompt_template.invoke({"topic": "고양이"})

ChatPromptValue(messages=[SystemMessage(content='당신은 주어진 요청에 도움을 주는 조수입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='고양이에 대한 농담을 해줘', additional_kwargs={}, response_metadata={})])

## 3.15 Anthropic Chat Model 예시

In [ ]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model='claude-3-5-sonnet-20241022')
llm.invoke([
    ("system", "당신은 주어진 요청에 도움을 주는 조수입니다."),
    ("user", "고양이에 대한 농담을 해줘")
])

## 3.16 Anthropic Chat Model 응답 예시

```
AIMessage(
    content=(
        "여기 몇 가지 재미있는 고양이 농담을 들려드릴게요:\n\n"
        "1. Q: 고양이가 가장 좋아하는 TV 프로그램은?\n"
        "   A: \"집사를 부탁해\"\n\n"
        "2. Q: 고양이가 인터넷을 하는 이유는?\n"
        "   A: 마우스를 잡으려고!\n\n"
        "3. Q: 고양이가 제일 싫어하는 강아지는?\n"
        "   A: 진돗개... 왜냐하면 \"진도\"(진도가) 너무 무서워서!\n\n"
        "4. Q: 고양이가 가장 좋아하는 운동은?\n"
        "   A: 캣치볼!\n\n"
        "5. Q: 고양이가 가장 좋아하는 음악은?\n"
        "   A: 캣(Cat) 스티븐스의 음악!\n\n"
        "어떠세요? 조금 억지스러울 수도 있지만, 고양이와 관련된 재미있는 말장난이에요!"
    ),
    additional_kwargs={},
    response_metadata={
        'id': 'msg_01YWroMa3sybmQEWVZis3tQ3',
        'model': 'claude-3-5-sonnet-20241022',
        'stop_reason': 'end_turn',
        'stop_sequence': None,
        'usage': {
            'input_tokens': 48,
            'output_tokens': 306
        }
    },
    id='run-368e3391-d27f-498f-8303-e3d6f6d8e2dc-0',
    usage_metadata={
        'input_tokens': 48,
        'output_tokens': 306,
        'total_tokens': 354,
        'input_token_details': {}
    }
)

```

## 3.17 Output Parser 를 사용하지 않는 일반적인 예시

In [ ]:
llm = ChatAnthropic(model='claude-3-5-sonnet-20241022')
response = llm.invoke([
    ("system", "당신은 주어진 요청에 대해 1개 문장 이내로 답변합니다."),
    ("user", "오늘 날씨가 어때?")
])

print(response.content)

## 3.18 StringOutputParser 를 이용하여 응답을 문자열로 변환

In [ ]:
from langchain_core.output_parsers import StrOutputParser

llm = ChatAnthropic(model='claude-3-5-sonnet-20241022')
chain = llm | StrOutputParser()
response = chain.invoke([
    ("system", "당신은 주어진 요청에 대해 1개 문장 이내로 답변합니다."),
    ("user", "오늘 날씨가 어때?")
])

print(response)

## 3.19 SimpleJsonOutputParser 를 이용하여 응답을 JSON으로 변환

In [ ]:
from langchain.output_parsers.json import SimpleJsonOutputParser

llm = ChatAnthropic(model='claude-3-5-sonnet-20241022')
chain = llm | SimpleJsonOutputParser()
response = chain.invoke([
    ("system", "당신은 질문에 답하는 `answer` 키가 포함된 JSON 객체를 반환합니다."),
    ("user", "오늘 날씨가 어때?")
])

print(type(response))
print(response)

## 3.20 파이프 연산자를 이용한 Langchain Chain 생성

In [ ]:
runnable1 = ChatPromptTemplate.from_messages([
    ("system", "주어진 질문에 대해 간단하고 명확한 답변을 제공해주세요."), 
    ("user", "{text}")
])

runnable2 = ChatAnthropic(model='claude-3-5-sonnet-20241022')

runnable3 = StrOutputParser()

chain = prompt_template | llm | StrOutputParser()

chain.invoke({"text": "안녕하세요. 오늘 날씨가 어때요?"})

## 3.21 고객 문의 분류 애플리케이션 초안 프롬프트 설계

```
당신은 고객 서비스 지원 챗봇입니다. 사용자가 제공한 텍스트를 아래 다섯 가지 카테고리 중 하나로 분류합니다. 

카테고리:
1. 환불 요청: 주문 취소 및 환불 관련 문의
2. 기술 지원: 로그인 문제, 기능 오류 등 기술적 문의
3. 계정 관리: 계정 생성, 비밀번호 변경 등 계정 관련 문의
4. 주문/배송 문의: 배송 현황, 주문 상태 관련 문의
5. 기타 문의: 위 카테고리에 해당하지 않는 문의

응답 지침:
- 출력은 반드시 한 가지 카테고리 이름(예: "환불 요청")만 반환해야 합니다.
- 출력에 불필요한 설명이나 추가 텍스트를 포함하지 않습니다.
```

## 3.22 고객 문의 분류 애플리케이션 프롬프트 템플릿

In [ ]:
from langchain.prompts.chat import ChatPromptTemplate

system_message = """당신은 고객 서비스 지원 챗봇입니다. 사용자가 제공한 텍스트를 아래 다섯 가지 카테고리 중 하나로 분류합니다. 

카테고리:
1. 환불 요청: 주문 취소 및 환불 관련 문의
2. 기술 지원: 로그인 문제, 기능 오류 등 기술적 문의
3. 계정 관리: 계정 생성, 비밀번호 변경 등 계정 관련 문의
4. 주문/배송 문의: 배송 현황, 주문 상태 관련 문의
5. 기타 문의: 위 카테고리에 해당하지 않는 문의

응답 지침:
- 출력은 반드시 한 가지 카테고리 이름(예: "환불 요청")만 반환해야 합니다.
- 출력에 불필요한 설명이나 추가 텍스트를 포함하지 않습니다.
3.23 고객 문의 분류 애플리케이션 프롬프트 템플릿
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_message),
    ("user", "{text}")
])


## 3.23 고객 문의 분류 애플리케이션 모델 정의

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_ollama import ChatOllama


# Claude 모델 설정
anthropic_llm = ChatAnthropic(
    model="claude-3-5-sonnet-20241022",
    temperature=0.01,
    max_tokens=10
)

# Mistral 모델 설정
mistral_llm = ChatOllama(
    model="mistral",
    temperature=0.01,
    max_tokens=10
)

## 3.24 고객 문의 분류 애플리케이션 출력 파서 정의

In [ ]:
from langchain_core.output_parsers import StrOutputParser


output_parser = StrOutputParser()

## 3.25 고객 문의 분류 애플리케이션 모델별 체인 정의 및 테스트

In [ ]:
anthropic_chain = prompt_template | anthropic_llm | output_parser
mistral_chain = prompt_template | mistral_llm | output_parser

text = {"text": "배송이 너무 지연되고 있어요."}

anthropic_result = anthropic_chain.invoke(text)
mistral_result = mistral_chain.invoke(text)

print("Anthropic 분류 결과:", anthropic_result)
print("Mistral 분류 결과:", mistral_result)